In [4]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [5]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wweth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wfth)),
])

nrmse_em, nrmse_diffs = em, diffs